# 📊 Dashboard de Ventas — Multi-Sucursal

Herramienta interactiva para analizar ventas de tiendas de construcción.
Funciona con **cualquier número de sucursales**: agrega filas nuevas al CSV
(con una columna `Sede` distinta) y el dashboard las reconoce automáticamente.

**Cómo usar:** ejecuta todas las celdas (`Cell > Run All` o ▶▶). Al final aparecen
los controles — cambia la sucursal, categoría o periodo y las gráficas y KPI se actualizan solas.

In [9]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import interactive_output, HBox, HTML as WHTML
from IPython.display import display, clear_output

pd.options.display.float_format = '{:,.2f}'.format

## 1. Cargar y preparar datos

Cambia `CSV_PATH` por la ruta de tu archivo cuando quieras usar datos actualizados.

In [10]:
CSV_PATH = "ventas.csv"

df = pd.read_csv(CSV_PATH)

# Limpieza y columnas calculadas
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Mes'] = df['Date'].dt.to_period('M').astype(str)
df['Anio'] = df['Date'].dt.year
df['Margen'] = df['Net Unit Price'] - df['Unit Cost']
df['Margen_pct'] = (df['Margen'] / df['Net Unit Price']) * 100
df['Descuento_pct'] = ((df['Price List'] - df['Net Unit Price']) / df['Price List']) * 100

print(f"{len(df):,} líneas | {df['Invoice No'].nunique():,} facturas | "
      f"{df['Sede'].nunique()} sucursales: {list(df['Sede'].unique())}")
print(f"Rango de fechas: {df['Date'].min().date()} a {df['Date'].max().date()}")
df.head()

1,009 líneas | 400 facturas | 2 sucursales: ['DEL', 'PAL']
Rango de fechas: 2025-01-01 a 2026-06-27


,Sede,Invoice No,Date,Line,Product Code,Invoice Quantity,Unit Cost,Net Unit Price,Price List,Price Limit,Price Std,Category Name,Total,Mes,Anio,Margen,Margen_pct,Descuento_pct
0,DEL,DEL1000,2025-05-12,10,PLY-680,2,11.45,21.20,23.32,20.14,21.20,Plywood,42.40,2025-05,2025,9.75,45.99,9.09
1,DEL,DEL1001,2025-08-17,10,CEM-425,1,23.54,34.87,38.36,33.13,34.87,Cemento,34.87,2025-08,2025,11.33,32.49,9.10
2,DEL,DEL1001,2025-08-17,20,PLY-994,1,10.94,19.58,21.54,18.60,19.58,Plywood,19.58,2025-08,2025,8.64,44.13,9.10
3,DEL,DEL1001,2025-08-17,30,TUB-134,2,33.42,58.77,64.65,55.83,58.77,Tuberías,117.54,2025-08,2025,25.35,43.13,9.10
4,DEL,DEL1002,2026-06-23,10,PIN-721,3,22.66,32.27,35.50,30.66,32.27,Pinturas,96.81,2026-06,2026,9.61,29.78,9.10


## 2. Controles interactivos

- **Sucursal / Categoría:** se llenan solos a partir de los datos.
- **Periodo:** puedes usar el selector rápido (todo el histórico, un año completo,
  últimos N meses) o elegir un mes exacto de inicio y fin en "Personalizado".

In [11]:
sede_options = ['Todas'] + sorted(df['Sede'].unique().tolist())
cat_options = ['Todas'] + sorted(df['Category Name'].unique().tolist())
meses_ordenados = sorted(df['Mes'].unique())
anios_disponibles = sorted(df['Anio'].dropna().unique().astype(int).tolist())

sede_w = widgets.Dropdown(options=sede_options, value='Todas', description='Sucursal:')
cat_w = widgets.Dropdown(options=cat_options, value='Todas', description='Categoría:')

# --- Selector rápido de periodo ---
presets = ['Todo el periodo']
presets += [f'Año {a} completo' for a in anios_disponibles]
presets += ['Últimos 3 meses', 'Últimos 6 meses', 'Últimos 12 meses', 'Personalizado']

preset_w = widgets.Dropdown(options=presets, value='Todo el periodo', description='Periodo rápido:',
                             style={'description_width': 'initial'})

# --- Selector personalizado (mes desde / hasta) ---
desde_w = widgets.Dropdown(options=meses_ordenados, value=meses_ordenados[0], description='Desde:')
hasta_w = widgets.Dropdown(options=meses_ordenados, value=meses_ordenados[-1], description='Hasta:')
custom_box = HBox([desde_w, hasta_w])
custom_box.layout.display = 'none'  # oculto salvo que se elija 'Personalizado'

def aplicar_preset(change=None):
    val = preset_w.value
    _ = change
    if val == 'Todo el periodo':
        desde_w.value, hasta_w.value = meses_ordenados[0], meses_ordenados[-1]
        custom_box.layout.display = 'none'
    elif val == 'Personalizado':
        custom_box.layout.display = 'flex'
    elif isinstance(val, str) and val.startswith('Año '):
        anio = int(val.split()[1])
        meses_anio = [m for m in meses_ordenados if m.startswith(str(anio))]
        desde_w.value, hasta_w.value = meses_anio[0], meses_anio[-1]
        custom_box.layout.display = 'none'
    elif isinstance(val, str) and val.startswith('Últimos '):
        n = int(val.split()[1])
        desde_w.value, hasta_w.value = meses_ordenados[-min(n, len(meses_ordenados))], meses_ordenados[-1]
        custom_box.layout.display = 'none'

preset_w.observe(aplicar_preset, names='value')

controles = HBox([sede_w, cat_w])
display(controles, preset_w, custom_box)

Dropdown(description='Periodo rápido:', options=('Todo el periodo', 'Año 2025 completo', 'Año 2026 completo', …

## 3. KPI y gráficas (se actualizan con los controles de arriba)

Si seleccionas **'Todas'** las sucursales, los KPI se muestran desglosados por sede y con una fila de **Total**.

In [12]:
out = widgets.Output()

def calcular_kpis(d):
    """Devuelve un dict con los KPI de un DataFrame ya filtrado."""
    ventas = d['Total'].sum()
    n_facturas = d['Invoice No'].nunique()
    ticket_prom = d.groupby('Invoice No')['Total'].sum().mean() if n_facturas else 0
    margen_pct = d['Margen_pct'].mean() if len(d) else 0
    descuento = d['Descuento_pct'].mean() if len(d) else 0
    unidades = d['Invoice Quantity'].sum()
    return {
        'Ventas': ventas, 'Facturas': n_facturas, 'Ticket prom.': ticket_prom,
        'Margen %': margen_pct, 'Descuento %': descuento, 'Unidades': unidades
    }

def filtrar(sede, categoria, desde, hasta):
    d = df.copy()
    if sede != 'Todas':
        d = d[d['Sede'] == sede]
    if categoria != 'Todas':
        d = d[d['Category Name'] == categoria]
    d = d[(d['Mes'] >= desde) & (d['Mes'] <= hasta)]
    return d

def render(sede, categoria, preset, desde, hasta):
    _ = preset
    with out:
        clear_output(wait=True)
        d = filtrar(sede, categoria, desde, hasta)

        if d.empty:
            print("No hay datos para este filtro.")
            return

        periodo_label = desde if desde == hasta else f"{desde} a {hasta}"
        display(WHTML(f"<div style='color:#666; font-size:13px; margin-bottom:8px;'>Periodo: <b>{periodo_label}</b></div>"))

        sedes_en_filtro = sorted(d['Sede'].unique())

        if sede == 'Todas' and len(sedes_en_filtro) > 1:
            # ---- Tabla de KPI por sede + Total ----
            filas = []
            for s in sedes_en_filtro:
                k = calcular_kpis(d[d['Sede'] == s])
                k['Sede'] = s
                filas.append(k)
            total_k = calcular_kpis(d)
            total_k['Sede'] = 'TOTAL'
            filas.append(total_k)

            tabla = pd.DataFrame(filas)[['Sede', 'Ventas', 'Facturas', 'Ticket prom.', 'Margen %', 'Descuento %', 'Unidades']]

            def formatear(row):
                es_total = row['Sede'] == 'TOTAL'
                estilo = "font-weight:bold; background:#eaeaea;" if es_total else ""
                return (f"<tr style='{estilo}'>"
                        f"<td style='padding:6px 12px;'>{row['Sede']}</td>"
                        f"<td style='padding:6px 12px;'>${row['Ventas']:,.2f}</td>"
                        f"<td style='padding:6px 12px;'>{row['Facturas']:,}</td>"
                        f"<td style='padding:6px 12px;'>${row['Ticket prom.']:,.2f}</td>"
                        f"<td style='padding:6px 12px;'>{row['Margen %']:,.1f}%</td>"
                        f"<td style='padding:6px 12px;'>{row['Descuento %']:,.1f}%</td>"
                        f"<td style='padding:6px 12px;'>{row['Unidades']:,}</td>"
                        f"</tr>")

            filas_html = "".join(tabla.apply(formatear, axis=1))
            tabla_html = f'''
            <table style="border-collapse:collapse; margin-bottom:16px; font-size:14px;">
              <thead>
                <tr style="background:#f4f4f4; text-align:left;">
                  <th style="padding:6px 12px;">Sucursal</th>
                  <th style="padding:6px 12px;">Ventas</th>
                  <th style="padding:6px 12px;">Facturas</th>
                  <th style="padding:6px 12px;">Ticket prom.</th>
                  <th style="padding:6px 12px;">Margen %</th>
                  <th style="padding:6px 12px;">Descuento %</th>
                  <th style="padding:6px 12px;">Unidades</th>
                </tr>
              </thead>
              <tbody>{filas_html}</tbody>
            </table>
            '''
            display(WHTML(tabla_html))
        else:
            # ---- Una sola sucursal (o solo hay una en los datos): tarjetas simples ----
            k = calcular_kpis(d)
            kpi_html = f'''
            <div style="display:flex; gap:16px; flex-wrap:wrap; margin-bottom:16px;">
              <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
                <div style="font-size:12px; color:#666;">VENTAS TOTALES</div>
                <div style="font-size:22px; font-weight:bold;">${k['Ventas']:,.2f}</div>
              </div>
              <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
                <div style="font-size:12px; color:#666;">FACTURAS</div>
                <div style="font-size:22px; font-weight:bold;">{k['Facturas']:,}</div>
              </div>
              <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
                <div style="font-size:12px; color:#666;">TICKET PROMEDIO</div>
                <div style="font-size:22px; font-weight:bold;">${k['Ticket prom.']:,.2f}</div>
              </div>
              <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
                <div style="font-size:12px; color:#666;">MARGEN % PROMEDIO</div>
                <div style="font-size:22px; font-weight:bold;">{k['Margen %']:,.1f}%</div>
              </div>
              <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
                <div style="font-size:12px; color:#666;">DESCUENTO PROMEDIO</div>
                <div style="font-size:22px; font-weight:bold;">{k['Descuento %']:,.1f}%</div>
              </div>
              <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
                <div style="font-size:12px; color:#666;">UNIDADES VENDIDAS</div>
                <div style="font-size:22px; font-weight:bold;">{k['Unidades']:,}</div>
              </div>
            </div>
            '''
            display(WHTML(kpi_html))

        # ---- Ventas por mes (y por sede si aplica) ----
        ventas_mes = d.groupby(['Mes', 'Sede'])['Total'].sum().reset_index()
        fig1 = px.line(ventas_mes, x='Mes', y='Total', color='Sede', markers=True,
                        title='Ventas por mes')
        fig1.update_layout(height=350, margin=dict(t=40, b=20))
        fig1.show()

        # ---- Ventas y margen por categoría ----
        cat_summary = d.groupby('Category Name').agg(
            Ventas=('Total', 'sum'),
            Margen_pct=('Margen_pct', 'mean')
        ).reset_index().sort_values('Ventas', ascending=False)

        fig2 = go.Figure()
        fig2.add_bar(x=cat_summary['Category Name'], y=cat_summary['Ventas'], name='Ventas ($)')
        fig2.add_scatter(x=cat_summary['Category Name'], y=cat_summary['Margen_pct'],
                          name='Margen %', yaxis='y2', mode='lines+markers', line=dict(color='crimson'))
        fig2.update_layout(
            title='Ventas por categoría (barras) vs. Margen % promedio (línea)',
            yaxis=dict(title='Ventas ($)'),
            yaxis2=dict(title='Margen %', overlaying='y', side='right'),
            height=380, margin=dict(t=40, b=20)
        )
        fig2.show()

        # ---- Mapa de calor: Margen % por Categoría y Sucursal ----
        if len(sedes_en_filtro) > 1:
            pivot_margen = d.pivot_table(index='Category Name', columns='Sede',
                                          values='Margen_pct', aggfunc='mean')
            # Ordenamos categorías de mayor a menor margen promedio para que se lea mejor
            pivot_margen = pivot_margen.loc[pivot_margen.mean(axis=1).sort_values(ascending=False).index]

            fig_heat = px.imshow(
                pivot_margen,
                text_auto=True,
                color_continuous_scale='RdYlGn',
                aspect='auto',
                labels=dict(x='Sucursal', y='Categoría', color='Margen %')
            )
            fig_heat.update_layout(
                title='Mapa de calor — Margen % por Categoría y Sucursal',
                height=max(320, 40 * len(pivot_margen) + 100),
                margin=dict(t=40, b=20)
            )
            fig_heat.show()

        # ---- Comparativo entre sucursales (si hay más de una en el filtro) ----
        if len(sedes_en_filtro) > 1:
            sede_summary = d.groupby('Sede').agg(Ventas=('Total', 'sum')).reset_index()
            fig3 = px.bar(sede_summary, x='Sede', y='Ventas', color='Sede',
                           title='Comparativo de ventas por sucursal', text_auto=True)
            fig3.update_layout(height=320, margin=dict(t=40, b=20), showlegend=False)
            fig3.show()

interactive_output(render, {'sede': sede_w, 'categoria': cat_w, 'preset': preset_w,
                             'desde': desde_w, 'hasta': hasta_w})
display(out)

Output()

## 4. Cómo escalar esto a más sucursales

- **Agregar una tienda nueva:** solo pega sus filas en el CSV con su propio código de `Sede` (por ejemplo `GYE`). No hay que tocar el código — los dropdowns y la tabla de KPI por sede se generan a partir de los datos.
- **Automatizar la carga:** si cada sucursal exporta su propio archivo, puedes concatenarlos con `pd.concat([pd.read_csv(f) for f in lista_de_archivos])` antes de la celda de limpieza.
- **Compartirlo como app (sin mostrar código):** instala `voila` (`pip install voila`) y corre `voila este_notebook.ipynb` — abre una página web con solo los controles y las gráficas, ideal para que lo use alguien que no usa Jupyter.
- **Actualización periódica:** si el CSV se sobrescribe cada semana/mes con datos nuevos, basta con volver a correr `Run All` para refrescar todo el dashboard.